In [2]:
import openeo
import os

import rasterio
from rasterio.mask import mask
from rasterio.merge import merge

import geopandas as gpd

from openeo.processes import if_, is_nodata

from collections import defaultdict

In [ ]:
# Remove connection to the backend and remove the refresh token 
#from openeo.rest.auth.config import RefreshTokenStore
#RefreshTokenStore().remove()

In [15]:
year = 2020

## 1) Input setting and preparation

In [16]:
start_date, end_date = ["2018-07-01", "2018-09-30"]

from pathlib import Path
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/" + str(year))
output_folder.mkdir(exist_ok=True)

In [ ]:
bursts_filename = '/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/S1_coh/s1_bursts_glacierSouthTyrol.geojson' # Extracted from the burst id map (https://sar-mpc.eu/sar-mpc-products/)
bursts = gpd.read_file(bursts_filename)
bursts.boundary.plot()

In [ ]:
url = "https://openeo.dataspace.copernicus.eu"
connection = openeo.connect(url).authenticate_oidc()

## 2) Multi-temporal mosaic of coherence

For each burst compute the InSAR coherence time series and then the multitemporal max

In [ ]:
import time

job_list = []

for idx, b in bursts.iterrows():

    datacube = connection.datacube_from_process(
        "sentinel1_sar_coherence",
        namespace="https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/sentinel1_sar_coherence.json",
        temporal_extent=[start_date, end_date],
        temporal_baseline=12,
        burst_id=int(b["burst_id"]),
        sub_swath=b["subswath_name"],
        coherence_window_az=7,
        coherence_window_rg=22,
        polarization="VV",
    )

    datacube = (
        datacube
        .reduce_dimension(dimension="t", reducer="max")
        .resample_spatial(projection=32632, resolution=10, method="bilinear")
        .save_result(format="GTiff")
    )

    job = datacube.create_job(
        title=f"sar_coherence_T{b['relative_orbit_number']}_{b['subswath_name']}_{b['burst_id']}_{year}"
    )

    job.start()
    job_list.append(job)

    print(f"Started {job.job_id}")

    time.sleep(10)   

## 3) Download the results

In [ ]:
from datetime import datetime
job_create_date = datetime.strptime('2026-10-07T10:00:00Z', "%Y-%m-%dT%H:%M:%SZ")
job_list = [connection.job(j['id']) for j in connection.list_jobs() if datetime.strptime(j['created'], "%Y-%m-%dT%H:%M:%SZ") > job_create_date]
#job_list = job_list[1:]
print(len(job_list), "jobs created after", job_create_date)
job_list

If in the same session where the jobs were launched, the results can be downloaded directly with:

In [ ]:
for i, j in enumerate(job_list):

    print(i, j.describe()["title"], j.status())

    if j.status() == "finished":

        title = j.describe()["title"]
        filepath = os.path.join(output_folder, f"{title}.tiff")

        print(f"Downloading: {filepath}")

        j.get_results().download_file(filepath)

        print("Done!")

### 4) Create composite per swadth

In [17]:
# Find all coherence files for each swadth
files = list(output_folder.glob("sar_coherence_T*.tiff"))

# Extract the track number from the filename
groups = defaultdict(list)

for f in files:
    parts = f.stem.split("_")
    
    track = parts[2]  
    
    groups[track].append(f)

# Create one composite for each track
for track, filenames in groups.items():

    print(f"{track}: {len(filenames)} files")

    datasets = [rasterio.open(f) for f in filenames]

    mosaic, transform = merge(
        datasets,
        method="first",
        nodata=0
    )

    profile = datasets[0].profile.copy()
    profile.update(
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        count=1,
        nodata=0,
        dtype=mosaic.dtype
    )

    output_file = output_folder / f"sar_coherence_{track}_composite.tiff"

    with rasterio.open(output_file, "w", **profile) as dst:
        dst.write(mosaic[0], 1)

    for src in datasets:
        src.close()

    print(f"{output_file}")

T117: 8 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2020/sar_coherence_T117_composite.tiff
T15: 4 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2020/sar_coherence_T15_composite.tiff
T95: 5 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2020/sar_coherence_T95_composite.tiff
T168: 6 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2020/sar_coherence_T168_composite.tiff
T44: 2 files
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2020/sar_coherence_T44_composite.tiff


### 5) Create mosaic from descending and ascending swadth

In [12]:
import re

files = list(output_folder.glob("sar_coherence_T*composite.tiff"))

ascending_orbits = [15, 44, 117]
descending_orbits = [95, 168]

# Separate files
ascending_files = []
descending_files = []

for f in files:

    # Extract T number from filename
    match = re.search(r"T(\d+)", f.name)

    if match:
        orbit = int(match.group(1))

        if orbit in ascending_orbits:
            ascending_files.append(f)

        elif orbit in descending_orbits:
            descending_files.append(f)

print("Ascending:")
for f in ascending_files:
    print(f)

print("\nDescending:")
for f in descending_files:
    print(f)

Ascending:
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/sar_coherence_T15_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/sar_coherence_T117_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/sar_coherence_T44_composite.tiff

Descending:
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/sar_coherence_T95_composite.tiff
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/sar_coherence_T168_composite.tiff


In [13]:
def merge_swaths(files, output_filename):

    datasets = [rasterio.open(f) for f in files]

    mosaic, transform = merge(
        datasets,
        method="max",
        nodata=0
    )

    meta = datasets[0].meta.copy()

    meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform,
        "count": 1,
        "nodata": 0
    })

    with rasterio.open(output_filename, "w", **meta) as dst:
        dst.write(mosaic)

    for ds in datasets:
        ds.close()

    print(f"Saved: {output_filename}")

In [14]:
ascending_filename = output_folder / f"coh_mosaic_ascending_{year}.tiff"
descending_filename = output_folder / f"coh_mosaic_descending_{year}.tiff"

merge_swaths(
    ascending_files,
    ascending_filename
)

merge_swaths(
    descending_files,
    descending_filename
)

Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/coh_mosaic_ascending_2026.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2026/coh_mosaic_descending_2026.tiff
